# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

In [1]:
import re, time
# Sur certains postes Windows, truststore permet à requests d'utiliser le magasin
# de certificats du système. Le notebook reste compatible s'il n'est pas installé.
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

# Sur certains postes Windows, truststore permet à requests d'utiliser le magasin
# de certificats du système. Le notebook reste compatible s'il n'est pas installé.
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [2]:
CONTACT = "https://github.com/DRIOUCH96"
ENTETES = {"User-Agent": f"TPFormation/1.0 ({CONTACT}) python-requests"}
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (https://github.com/DRIOUCH96) python-requests'}


In [13]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (https://github.com/DRIOUCH96) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [14]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES, timeout=20)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser()
    regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [15]:
regles = lire_robots(SITE)
url_page_2 = urljoin(SITE, "/page/2/")
url_einstein = urljoin(SITE, "/author/Albert-Einstein")

autorise_pages = True if regles is None else regles.can_fetch(
    ENTETES["User-Agent"], url_page_2
)
autorise_auteur = True if regles is None else regles.can_fetch(
    ENTETES["User-Agent"], url_einstein
)

https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée


In [16]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [17]:
# Collecter chaque page une fois, jusqu'à l'absence de lien « Next ».
lignes_citations = []
nb_requetes = 0
url = SITE + "/"

while url:
    if regles is not None and not regles.can_fetch(ENTETES["User-Agent"], url):
        raise PermissionError("robots.txt interdit : " + url)

    reponse = get_poli(url)
    nb_requetes += 1
    reponse.raise_for_status()
    page = BeautifulSoup(reponse.content, "html.parser")

    for bloc in page.select("div.quote"):
        lien_auteur = bloc.select_one("a[href^='/author/']")
        texte = bloc.select_one("span.text").get_text(strip=True).strip('“”"')
        lignes_citations.append({
            "texte": texte,
            "auteur": bloc.select_one("small.author").get_text(strip=True),
            "tags": [tag.get_text(strip=True) for tag in bloc.select("a.tag")],
            "url_page": url,
            "url_auteur": urljoin(url, lien_auteur["href"]),
        })

    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None

citations = pd.DataFrame(lignes_citations)
print(f"{len(citations)} citations collectées sur {citations['url_page'].nunique()} pages")
citations.head()

100 citations collectées sur 10 pages


,texte,auteur,tags,url_page,url_auteur
0,The world as we have created it is a process o...,Albert Einstein,"[change, deep-thoughts, thinking, world]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/Albert-Eins...
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"[abilities, choices]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/J-K-Rowling
2,There are only two ways to live your life. One...,Albert Einstein,"[inspirational, life, live, miracle, miracles]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/Albert-Eins...
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"[aliteracy, books, classic, humor]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/Jane-Austen
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"[be-yourself, inspirational]",https://quotes.toscrape.com/,https://quotes.toscrape.com/author/Marilyn-Monroe


In [18]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [19]:
# Dédupliquer les URL avant la boucle évite de redemander une même fiche.
fiches_auteurs = []
urls_auteurs_uniques = citations["url_auteur"].drop_duplicates()

for url_auteur in urls_auteurs_uniques:
    if regles is not None and not regles.can_fetch(ENTETES["User-Agent"], url_auteur):
        raise PermissionError("robots.txt interdit : " + url_auteur)

    reponse = get_poli(url_auteur)
    nb_requetes += 1
    reponse.raise_for_status()
    page = BeautifulSoup(reponse.content, "html.parser")

    fiches_auteurs.append({
        "url_auteur": url_auteur,
        "auteur": page.select_one("h3.author-title").get_text(strip=True),
        "date_naissance": page.select_one("span.author-born-date").get_text(strip=True),
        "lieu_naissance": page.select_one("span.author-born-location").get_text(strip=True).removeprefix("in "),
        "biographie": page.select_one("div.author-description").get_text(" ", strip=True),
    })

auteurs = pd.DataFrame(fiches_auteurs)
print(f"{len(auteurs)} auteurs uniques ; {nb_requetes} requêtes au total")
auteurs.head()

50 auteurs uniques ; 60 requêtes au total


,url_auteur,auteur,date_naissance,lieu_naissance,biographie
0,https://quotes.toscrape.com/author/Albert-Eins...,Albert Einstein,"March 14, 1879","Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,https://quotes.toscrape.com/author/J-K-Rowling,J.K. Rowling,"July 31, 1965","Yate, South Gloucestershire, England, The Unit...",See also: Robert GalbraithAlthough she writes ...
2,https://quotes.toscrape.com/author/Jane-Austen,Jane Austen,"December 16, 1775","Steventon Rectory, Hampshire, The United Kingdom",Jane Austen was an English novelist whose work...
3,https://quotes.toscrape.com/author/Marilyn-Monroe,Marilyn Monroe,"June 01, 1926",The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...
4,https://quotes.toscrape.com/author/Andre-Gide,André Gide,"November 22, 1869","Paris, France",André Paul Guillaume Gide was a French author ...


In [20]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 60 requêtes pour 60 attendues au minimum : fiches non redemandées


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [21]:
citations_completes = citations.merge(
    auteurs.drop(columns="auteur"),
    on="url_auteur",
    how="left",
    validate="many_to_one",
)
top_auteurs = citations["auteur"].value_counts().head(5)
top_tags = citations.explode("tags")["tags"].value_counts().head(10)

print("Top 5 des auteurs :")
display(top_auteurs)
print("Top 10 des tags :")
display(top_tags)

# Fichiers intermédiaires utiles pour contrôler ou réutiliser la mission.
citations.to_csv("citations.csv", index=False, encoding="utf-8")
auteurs.to_csv("auteurs.csv", index=False, encoding="utf-8")
citations_completes.to_csv("citations_completes.csv", index=False, encoding="utf-8")

Top 5 des auteurs :


auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64

Top 10 des tags :


tags
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64

In [22]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)

> Comme pour Books to Scrape, j'ai parcouru la pagination avec `requests`, analysé le HTML avec BeautifulSoup et transformé les lignes en DataFrame.  
> La structure diffère : une citation a plusieurs tags et son auteur renvoie vers une fiche partagée par plusieurs citations.  
> J'ai donc adapté les sélecteurs CSS, retiré les guillemets du texte et conservé les tags dans une liste.  
> J'ai dédupliqué les URL d'auteurs avant les requêtes, puis effectué une fusion plusieurs-vers-un.  
> Un changement des classes, des liens de pagination ou des champs de fiche casserait les sélecteurs et imposerait de nouveaux contrôles.
